# LULC

This notebook does 3 things:
1. Generate training data for LULC classification. It uses 3 existing LULC products.
2. Filter training data per class using kmeans clustering to exclude outliers.
3. Train a random forest model, and then predict LULC.

### Steps: 
1. Training data:
For each site extract training data.
  -> Find product agreement. mask to gadm 100m buffer.
  -> Add geomad, indices, elevation etc.
  -> Write csv per site of training data.
do this in a notebook. don't commit training data.

1a. Filter outliers per class.

2. Train model:
Try one model for all sites. (append all CSVs)
export model dump (python pickle?)
in future we may need to make different models for different regions and year ranges.
train the model using the geomad of the year of the input products.

3. Predict:
per grid tile:
  per year:
    load geomad/indices/elevation etc.
    make using get_gadm (buffered 100m)
    predict
This is as a command. 

### Load packages


In [10]:
# Not sure if this is needed.
# Reload functions during development
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [ ]:
%matplotlib inline

import json

# Scientific core
import numpy as np
import pandas as pd

# Visualisation
import matplotlib.gridspec as gridspec
from matplotlib import pyplot as plt
from matplotlib.colors import BoundaryNorm, ListedColormap
from matplotlib.patches import Patch

# Geospatial
import rasterio
import xarray as xr
from ipyleaflet import basemaps
from odc.geo.xr import assign_crs
from odc.stac import load
from planetary_computer import sign_url
from pystac.client import Client

# Machine learning
from scipy.ndimage import minimum_filter
from scipy.spatial.distance import cdist
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import ConfusionMatrixDisplay, classification_report, confusion_matrix, silhouette_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Local
from ldn.grids import get_gadm
from ldn.random_sampling import random_sampling
from ldn.typology import classes, colors, world_cover_map, cci_lc_map, io_map
from ldn.utils import get_geomad_dem_indices
from notebooks.src.Compare_LULC_func import standardise_class

In [4]:
# Parameters
datetime = '2020-06' # TODO: Just use 2020?
datetime_year = datetime.split("-")[0]
lulc_resolution_wc_io = 10
lulc_resolution_cci = 300
# lulc_resolution = 30 # meters. CCI: 300m, WC: 10m, IO: 10m. Meet in the middle with 30m. TODO: Load native? Or load all at 10 for best granularity.
lulc_class_raster = 'lulc_agree.tif'
analysis_crs = 'EPSG:6933' # Equal Earth for global analysis
output_crs = 'EPSG:4326' # WGS84
# output_crs = 'EPSG:32648' # UTM Zone 48N for Singapore
class_attr='lulc'
buffer_m  = 100
# Use Planetary Computer STAC API
catalog = Client.open("https://planetarycomputer.microsoft.com/api/stac/v1/")
country_of_interest = {"Singapore": "SGP"}
# country_of_interest = {"Fiji": "FJI"}

In [5]:
# Get the country boundary and buffer it to get include coastal areas.
country = get_gadm(countries=country_of_interest, overwrite=True)
print(f"Country CRS: {country.crs}. Reprojecting to analysis CRS: {analysis_crs} and buffering by {buffer_m} meters.")
country = country.to_crs(analysis_crs)
country = country.buffer(buffer_m)

Country CRS: EPSG:4326. Reprojecting to analysis CRS: EPSG:6933 and buffering by 100 meters.


In [6]:
assert len(country) == 1, "Expected only one geometry after buffering, but got multiple."
country.explore()

In [12]:
country_geomad_dem = get_geomad_dem_indices(country, datetime_year, analysis_crs=analysis_crs ,catalog=catalog)
# This takes ~1.5min for Singapore.

Found 2 STAC documents for this AOI and year
Parsed 2 STAC items
Available bands: ['green', 'smad', 'swir16', 'bcmad', 'count', 'red', 'blue', 'swir22', 'nir08', 'emad']
Native GeoMAD dataset CRS: 6933
GeoMAD dataset shape: FrozenMappingWarningOnValuesAccess({'y': 1304, 'x': 1541})
<xarray.Dataset> Size: 3kB
Dimensions:      (y: 5, x: 5)
Coordinates:
  * y            (y) float64 40B 1.878e+05 1.877e+05 ... 1.877e+05 1.877e+05
  * x            (x) float64 40B 9.997e+06 9.997e+06 ... 9.997e+06 9.997e+06
    spatial_ref  int32 4B 6933
    time         datetime64[ns] 8B 2020-01-01
Data variables: (12/18)
    green        (y, x) float64 200B 0.05352 0.0904 0.05591 ... 0.05077 0.06128
    smad         (y, x) float32 100B 0.0006227 0.000318 ... 0.0005286 0.0007699
    swir16       (y, x) float64 200B 0.162 0.1841 0.1585 ... 0.1495 0.1907
    bcmad        (y, x) float32 100B 0.02033 0.0157 0.01937 ... 0.02001 0.02011
    count        (y, x) uint16 50B 12 12 13 13 13 12 12 ... 13 14 13 12 13 13

In [ ]:
# This cell is just to check the data visually.
for column in country_geomad_dem.data_vars:
    # print(f"{column}: {country_geomad_dem[column].shape}, {country_geomad_dem[column].dtype}")
    print(f"Column: {column}, min: {country_geomad_dem[column].min().values}, max: {country_geomad_dem[column].max().values}, dtype: {country_geomad_dem[column].dtype}")

# test_attr = "slope"
# test_attr = "elevation"
# test_attr = "aspect"
# test_attr = "ndvi"
test_attr = "red"
test_attr_colormap = {"elevation": "terrain", "aspect": "viridis", "slope": "plasma", "ndvi": "YlGn", "red": "Reds"}[test_attr]

country_geomad_dem[test_attr].odc.explore(
    cmap=test_attr_colormap,
    vmin=country_geomad_dem[test_attr].min().values,
    vmax=country_geomad_dem[test_attr].max().values,
)

## Use 3 LU/LC products. Find agreement between them.

TODO: Experiment with data quality bands - some LULC datasets have quality information which could be used as additional filters.

In [ ]:
# Steps: TODO: this.
# 1. load the highest res product (there are 2 at 10m.)
# 2. load the others to be like the 1st. use nearest resampling to avoid creating new classes that may not exist in the original data.
# 3. find agreemet where values are exact. neighbours also agree.


country_geojson = json.loads(country.to_json(to_wgs84=True))["features"][0]["geometry"]

# A generic function to load a product given the boundary
def load_lulc_data(product: str, resolution = lulc_resolution_wc_io):
    items = catalog.search(
        collections=[product],
        intersects=country_geojson,
        datetime=datetime,
    ).item_collection()

    print(f"Found {len(items)} items for product {product}")

    ds = load(
        items,
        intersects=country_geojson,
        crs=analysis_crs, # Use native? Need to align products on CRS/grid
        groupby="solar_day",
        resolution=resolution, # TODO: Load native?
        chunks={"x": 2048, "y": 2048}, # Needed?
        patch_url=sign_url,
        resampling="nearest",
    )

    print(f"Product {product} has variables: {ds.data_vars}")
    return ds

In [ ]:
cci_30m = load_lulc_data('esa-cci-lc', resolution=lulc_resolution_cci) # 300m is native. TODO: Load native?
vars = list(cci_30m.data_vars)
cci_30m["esa_cci_lc"] = cci_30m["lccs_class"]
ds_tmp = cci_30m.drop_vars(vars)
cci_30m = ds_tmp.squeeze().load() # removes any singleton dimension and then load the full data

# Clip to AOI. ODC STAC Load intersects param doesn't clip.
# TODO: Country is already in 6933. Products are 6933 too, but might load native in future?
cci_30m = cci_30m.rio.clip(country.to_crs(analysis_crs).geometry, crs=analysis_crs)

In [ ]:
cci_30m["esa_cci_lc"].odc.explore()
# cci_30m["esa_cci_lc"]

In [ ]:
# TODO: Make this a function because we do it twice.
wc_30m = load_lulc_data('esa-worldcover', resolution=lulc_resolution_wc_io) # 10m is native. # TODO: Load native?
vars = list(wc_30m.data_vars)
wc_30m["esa_worldcover"] = wc_30m["map"]
ds_tmp = wc_30m.drop_vars(vars)
wc_30m = ds_tmp.squeeze().load() #removes any singleton dimension and then load the full data

# Clip to AOI. ODC STAC Load intersects param doesn't clip.
# TODO: Country is already in 6933. Products are 6933 too, but might load native in future?
wc_30m = wc_30m.rio.clip(country.to_crs(analysis_crs).geometry, crs=analysis_crs)

# Ensure alignment.
# TODO: if reprojecting, use nearest!
# TODO: Don't reproject. It should be loaded correctly in the first place.
# TODO: Just load like highest res product.
wc_30m["esa_worldcover"] = wc_30m["esa_worldcover"].rio.reproject_match(
    cci_30m["esa_cci_lc"], resampling=rasterio.enums.Resampling.mode  # majority vote (the value that appears most often)
)

In [ ]:
# wc_30m["esa_worldcover"].odc.explore()
wc_30m["esa_worldcover"]

In [ ]:
# TODO: Make this a function because we do it twice.
io_30m = load_lulc_data('io-lulc-annual-v02', resolution=lulc_resolution_wc_io) # 10m is native. # TODO: Load native?
vars = list(io_30m.data_vars)
io_30m["io_lulc"] = io_30m["data"]
ds_tmp = io_30m.drop_vars(vars)
io_30m = ds_tmp.squeeze().load() # removes any singleton dimension and then load the full data

# Clip to AOI. ODC STAC Load intersects param doesn't clip.
# TODO: Country is already in 6933. Products are 6933 too, but might load native in future?
io_30m = io_30m.rio.clip(country.to_crs(analysis_crs).geometry, crs=analysis_crs)

# Ensure alignment.
# TODO: if reprojecting, use nearest!
# TODO: Don't reproject. It should be loaded correctly in the first place.
# TODO: Just load like highest res product.
io_30m["io_lulc"] = io_30m["io_lulc"].rio.reproject_match(
    cci_30m["esa_cci_lc"], resampling=rasterio.enums.Resampling.mode  # majority vote (the value that appears most often)
)

In [ ]:
io_30m["io_lulc"].odc.explore()

In [ ]:
# Standardise the classes to match the typology
cci_30m['esa_cci_lc'] = standardise_class(cci_30m['esa_cci_lc'], cci_lc_map)
wc_30m['esa_worldcover'] = standardise_class(wc_30m['esa_worldcover'], world_cover_map)
io_30m['io_lulc'] = standardise_class(io_30m['io_lulc'], io_map)
wc_30m

# Create layer where all 3 land cover products agree.

In [ ]:
# TODO: Now products are in native resolution. They won't align.

# agreed_class = Requires exact 3-way match at every pixel
# agreed_class_strict = Exact match + all 8 neighbours also match (3×3 minimum_filter erosion)

# Valid pixels — all three products have data
valid = (wc_30m['esa_worldcover'] > 0) & (cci_30m['esa_cci_lc'] > 0) & (io_30m['io_lulc'] > 0)

# 1. Strict pixel-level agreement (all 3 products match exactly)
all_agree = (
    (wc_30m['esa_worldcover'] == cci_30m['esa_cci_lc']) &
    (cci_30m['esa_cci_lc'] == io_30m['io_lulc'])
) & valid  # boolean, no NaN

agreed_class = wc_30m['esa_worldcover'].where(all_agree, other=0).rename(class_attr)

# 2. Strict agreement WITH neighbours also fully agreeing
# Erode the agreement mask by 1 pixel: a pixel passes only if it AND all
# 8 surrounding pixels also agree (minimum filter on boolean → 1 iff all 1).
agree_f = all_agree.values.astype("float32")
# minimum_filter: every pixel takes the minimum of its 3×3 neighbourhood
neighbour_agree_mask = minimum_filter(agree_f, size=3, mode="constant", cval=0) == 1

agreed_class_strict = wc_30m['esa_worldcover'].where(
    xr.DataArray(neighbour_agree_mask, coords=all_agree.coords, dims=all_agree.dims),
    other=0
).rename(f"{class_attr}_strict")

# Diagnostics


for name, layer in [("pixel agreement", agreed_class), ("neighbour-strict agreement", agreed_class_strict)]:
    _classes, counts = np.unique(layer.values, return_counts=True)
    counts_df = pd.DataFrame({"class": _classes, "pixel_count": counts})
    print(f"\n=== {name} ===")
    print(counts_df)

In [ ]:
# Write agreeing pixels as tiff.
agreed_class.odc.write_cog(lulc_class_raster)
agreed_class_strict.odc.write_cog(lulc_class_raster.replace(".tif", "_strict.tif"))

classification_map = agreed_class
# classification_map = agreed_class_strict
classification_map

# Visualise the agreeing classes from 3 products

In [ ]:
fig, axes = plt.subplots(1,1)

# Plot classification map
unique_values=np.unique(classification_map)
cmap=ListedColormap([colors[k] for k in unique_values])
norm = BoundaryNorm(list(unique_values)+[np.max(unique_values)+1], cmap.N)
classification_map.plot.imshow(ax=axes, 
                   cmap=cmap,
                   norm=norm,
                   add_labels=True, 
                   add_colorbar=False,
                   interpolation='none')
# add colour legend
patches_list=[Patch(facecolor=colour) for colour in colors.values()]
axes.legend(patches_list, list(classes.keys()),loc='upper center', ncol =4, bbox_to_anchor=(0.5, -0.1))

## Generate random training samples
We generate some randomly distributed samples for each class from the clipped classification map using the `random_sampling` function. This function takes in a few parameters:  
* `da`: a classified map in the format of 2-dimensional xarray.DataArray
* `n`: total number of points to sample.
* `min_sample_n`: Minimum number of samples to generate per class if proportional number is smaller than this. **Note that the resultant number of samples may be higher than the set `n` due to setting of this minimum number of samples.** 
* `sampling`: the sampling strategy, e.g. 'stratified_random' where each class has a number of points proportional to its relative area, 'equal_stratified_random' where each class has the same number of points, or 'manual' which allows you to define number of samples for each class.
* `out_fname`: a filepath name for the function to export a shapefile/geojson of the sampling points into a file. You can set this to `None` if you don't need to output the file.
* `class_attr`: This is the column name of output dataframe that contains the integer class values on the classified map. 
* `drop_value`: pixel value on the classification map to be excluded from sampling.

The output of the function is a geopandas dataframe of randomly distributed points containing a column `class_attr` identifying class values. 

Here we extract around 1000 training points in total and export the points in a geojson file for use in the rest of workflow. Here we use the stratified sampling method by setting 'equal_stratified_random', but also set the minimum number of samples as 75 to avoid missing samples for some minor classes. 

As mentioned earlier we don't want the abandoned classes to be included in the samples we set drop_value as 0 before implementing the function:

In [ ]:
# Reproject to WGS84 so sample points are in lat/lon
# TODO: Don't reproject. It should be loaded correctly in the first place. Maybe all product data should be loaded in WGS84 to begin with? Or I can tweak random_sampling to accept other CRSs.
# TODO: Check the native CRS of the 3 products.
# TODO: Reprojecting is a red flag. It can cause a lot of issues. At least use nearest neighbour to avoid creating new classes that may not exist in the original data.
classification_map_4326 = classification_map.rio.reproject(output_crs, resampling=rasterio.enums.Resampling.nearest)
classification_map_4326.head()

In [ ]:
out_fname='training_data.geojson'
n=2100
min_sample_n=300
drop_value=0
gpd_random_samples=random_sampling(da=classification_map_4326,n=n,sampling='stratified_random',
                                   min_sample_n=min_sample_n,out_fname=None,class_attr=class_attr,drop_value=drop_value)

In [ ]:
print(f"CRS: {gpd_random_samples.crs}")
gpd_random_samples.head()

## Visualise the training data by class

In [ ]:
gpd_random_samples.explore(
    column=class_attr,
    categorical=True,
    categories=(present_classes := sorted(gpd_random_samples[class_attr].unique())),
    cmap=[colors[c] for c in present_classes],
    legend=True,
    style_kwds={"radius": 6, "fillOpacity": 0.8, "weight": 0.5}
)

## Extract Geomedian/GeoMAD values at training data point locations.

For now we run and load individual tiles and mosaic them because we have not run all tiles yet.

Once we have finalised the GeoMAD data, we will set up a GeoParquet STAC index for it. Then querying it by bbox will be simple and replace the more complicated process implemented here.

In [ ]:
country_geomad_dem = get_geomad_dem_indices(country.geometry, datetime_year, analysis_crs=analysis_crs ,catalog=catalog)

In [ ]:
geomad_dem_inidices.odc.explore()

In [ ]:
band_names_with_indices = [v for v in geomad_dem_inidices.data_vars]
print(band_names_with_indices)

# Extract Geomedian and GeoMAD band values at each sample point location

# Ensure both datasets are in the same CRS (6933)
# TODO: Reproject sample points to match native CRS of geomad before sampling.
gpd_random_samples_6933 = gpd_random_samples.to_crs(analysis_crs)

# Build point-wise x/y arrays with the SAME 'points' index
points_idx = np.arange(len(gpd_random_samples_6933))
xs = xr.DataArray(gpd_random_samples_6933.geometry.x.values, dims="points", coords={"points": points_idx})
ys = xr.DataArray(gpd_random_samples_6933.geometry.y.values, dims="points", coords={"points": points_idx})
# Point-wise nearest sampling (one sampled pixel per input point)
sampled = geomad_dem_inidices[band_names_with_indices].sel(x=xs, y=ys, method="nearest")

# Convert to dataframe indexed by points; keep exactly one row per point
sampled_df = sampled.to_dataframe().reset_index()
sampled_df = sampled_df.sort_values("points").reset_index(drop=True)

# Merge back to training points (original CRS)
training_data = gpd_random_samples.reset_index(drop=True).copy()
training_data = pd.concat([training_data, sampled_df[band_names_with_indices]], axis=1)

print(f"Training data shape: {training_data.shape}")
print("Unique values per band:")
print(training_data[band_names_with_indices].nunique())
training_data.head()

## Visualise GeoMedian red values per sample point

In [ ]:
# Visualise red values using the actual data range
training_data.explore(
    column="red",
    cmap="Reds",
    k=7,
    scheme="natural_breaks",
    legend=True,
    style_kwds={"radius": 4, "fillOpacity": 0.8, "weight": 0.5},
)

## Explore elevation data on training data points

In [ ]:
training_data.explore(
    column="elevation",
    cmap="Blues",
    scheme="natural_breaks",
    k=7,
    legend=True,
    style_kwds={"radius": 4, "fillOpacity": 0.8, "weight": 0.5},
)

## Explore NDVI data on training data points

In [ ]:
training_data.explore(
    column="ndvi",
    cmap="Greens",
    scheme="quantiles",
    k=7,
    legend=True,
    style_kwds={"radius": 4, "fillOpacity": 0.8, "weight": 0.5},
)

In [ ]:
out_data = training_data.drop(columns=["time", 'spatial_ref', 'count'])

# Write the final training data
out_data.to_file(out_fname, driver="GeoJSON", index=False)
# out_data.to_file(out_fname.replace(".geojson", ".gpkg"), driver="GPKG")
# out_data.to_csv(out_fname.replace(".geojson", ".csv"), index=False)

print(f"Saved training data as geojson, CSV, and geopackage to {out_fname}")

# Section 2: Filter training data by K-Means clustering
Goal is to reduce confusion. Make training point classes clean/accurate/spectrally separable.

In [ ]:
print(training_data.columns)

training_data.explore(
    column=class_attr,
    categorical=True,
    categories=(present_classes := sorted(training_data[class_attr].unique())),
    cmap=[colors[c] for c in present_classes],
    legend=True,
    style_kwds={"radius": 6, "fillOpacity": 0.8, "weight": 0.5}
)

In [ ]:
# 1. Prep feature matrix ────────────────────────────────────────────────────
exclude_cols = ['lulc', 'geometry']
feature_cols = [c for c in training_data.columns if c not in exclude_cols]

training_data['outlier'] = False
training_data.loc[training_data[feature_cols].isna().any(axis=1), 'outlier'] = True

nan_rows = training_data['outlier'].sum()
print(f"Rows with NaNs (auto-flagged): {nan_rows}")

valid_mask = ~training_data[feature_cols].isna().any(axis=1)
valid_idx  = training_data.index[valid_mask]

X = training_data.loc[valid_idx, feature_cols].values
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# 2. Visualisation helpers ──────────────────────────────────────────────────
CLUSTER_CMAP = 'tab10'

def plot_pca(ax, Xc, labels, centers, outlier_mask, class_name, best_score):
    """PCA scatter — inliers coloured by cluster, outliers as red crosses."""
    pca      = PCA(n_components=2)
    X_2d     = pca.fit_transform(Xc)
    c_2d     = pca.transform(centers)
    var      = pca.explained_variance_ratio_
    k        = len(np.unique(labels))
    inliers  = ~outlier_mask

    ax.scatter(
        X_2d[inliers, 0], X_2d[inliers, 1],
        c=labels[inliers], cmap=CLUSTER_CMAP, vmin=0, vmax=max(k - 1, 1),
        alpha=0.55, s=25, linewidths=0, label='inlier'
    )
    if outlier_mask.any():
        ax.scatter(
            X_2d[outlier_mask, 0], X_2d[outlier_mask, 1],
            c='red', marker='x', s=50, linewidths=1.2, label='outlier'
        )
    ax.scatter(
        c_2d[:, 0], c_2d[:, 1],
        c='black', marker='*', s=180, zorder=5, label='centroid'
    )
    ax.set(
        xlabel=f'PC1 ({var[0]:.1%})',
        ylabel=f'PC2 ({var[1]:.1%})',
        title=f'PCA  |  k={k}  sil={best_score:.3f}'
    )
    ax.legend(fontsize=7, markerscale=0.9)


def plot_heatmap(ax, centers, feature_cols, outlier_count, n):
    """Centroid heatmap — rows = clusters, cols = features."""
    im = ax.imshow(centers, aspect='auto', cmap='RdBu_r')
    ax.set_xticks(range(len(feature_cols)))
    ax.set_xticklabels(feature_cols, rotation=45, ha='right', fontsize=7)
    ax.set_yticks(range(len(centers)))
    ax.set_yticklabels([f'C{i}' for i in range(len(centers))], fontsize=8)
    plt.colorbar(im, ax=ax, label='Scaled value', pad=0.02)

    # Annotate each cell with the scaled value
    for r in range(centers.shape[0]):
        for c in range(centers.shape[1]):
            ax.text(c, r, f'{centers[r, c]:.1f}',
                    ha='center', va='center', fontsize=6,
                    color='white' if abs(centers[r, c]) > 1.2 else 'black')

    ax.set_title(f'Centroids  |  outliers={outlier_count} ({100*outlier_count/n:.1f}%)')


# 3. Per-class clustering + outlier flagging + visualisation ────────────────
threshold = 2.0   # ← tune: lower = more aggressive flagging

classes = training_data.loc[valid_idx, 'lulc'].unique()

for cls in classes:
    mask = (training_data.loc[valid_idx, 'lulc'] == cls).values
    idx  = valid_idx[mask]
    Xc   = X_scaled[mask]
    n    = len(Xc)

    if n < 6:
        continue
    k_max = min(5, n // 5)
    if k_max < 2:
        continue

    # Optimal k via silhouette ──────────────────────────────────────────
    best_k, best_score = 2, -1
    for k in range(2, k_max + 1):
        km     = KMeans(n_clusters=k, random_state=42, n_init=10)
        labels = km.fit_predict(Xc)
        score  = silhouette_score(Xc, labels)
        if score > best_score:
            best_k, best_score = k, score

    km_final = KMeans(n_clusters=best_k, random_state=42, n_init=10).fit(Xc)
    labels   = km_final.labels_
    centers  = km_final.cluster_centers_

    # Outlier flagging ──────────────────────────────────────────────────
    outlier_mask = np.zeros(n, dtype=bool)
    for cluster_id in range(best_k):
        in_cluster = labels == cluster_id
        pts        = Xc[in_cluster]
        dists      = cdist(pts, centers[cluster_id].reshape(1, -1)).flatten()
        median_d   = np.median(dists)
        if median_d == 0:
            continue
        flagged = dists > threshold * median_d
        outlier_mask[np.where(in_cluster)[0][flagged]] = True

    training_data.loc[idx[outlier_mask], 'outlier'] = True

    print(f"  {str(cls):30s} | n={n:4d} | k={best_k} | sil={best_score:.3f} "
          f"| outliers={outlier_mask.sum():4d} ({100*outlier_mask.mean():.1f}%)")

    # Figure: PCA (left) + heatmap (right) ─────────────────────────────
    fig = plt.figure(figsize=(14, 4.5))
    fig.suptitle(f'Class: {cls}  (n={n})', fontsize=12, fontweight='bold', y=1.01)

    gs  = gridspec.GridSpec(1, 2, width_ratios=[1, 1.6], figure=fig)
    ax_pca  = fig.add_subplot(gs[0])
    ax_heat = fig.add_subplot(gs[1])

    plot_pca(ax_pca, Xc, labels, centers, outlier_mask, cls, best_score)
    plot_heatmap(ax_heat, centers, feature_cols, outlier_mask.sum(), n)

    plt.tight_layout()
    plt.show()

# 4. Summary ────────────────────────────────────────────────────────────────
print(f"\nTotal points : {len(training_data):,}")
print(f"Clean        : {(~training_data['outlier']).sum():,}")
print(f"Outliers     : {training_data['outlier'].sum():,}")
print(f"\nOutliers per class:")
print(training_data.groupby('lulc')['outlier'].mean().mul(100).round(1).astype(str) + '%')
training_clean = training_data[~training_data['outlier']]
training_clean.explore(
    column=class_attr,
    categorical=True,
    categories=(present_classes := sorted(training_clean[class_attr].unique())),
    cmap=[colors[c] for c in present_classes],
    legend=True,
    style_kwds={"radius": 6, "fillOpacity": 0.8, "weight": 0.5}
)

In [ ]:
# Write the data with the new outlier column for use training a model and prediction.
training_data.to_file("training_data.geojson", driver="GeoJSON")
# TODO:
# Look at how Other class is clustered. Maybe we can split Other to spectrally distinct classes that are merged after prediction. Other is getting about 6% accuracy.
# For land cover / remote sensing data, PCA + centroid heatmap together give the most actionable insight — PCA shows separation, the heatmap explains it in terms of your original bands/indices.

# Section 3: Train and Predict

In [ ]:
# TODO: This is done twice.
training_data = training_data[~training_data['outlier']] # Remove outliers from the training data
training_data.drop(columns=['outlier'], inplace=True) # Drop outlier column as it's no longer needed
# Remove geometry column for train/test split
training_data = training_data.drop(columns="geometry")

print(len(training_data))

training_data.explore(
    column=class_attr,
    categorical=True,
    categories=(present_classes := sorted(training_data[class_attr].unique())),
    cmap=[colors[c] for c in present_classes],
    legend=True,
    style_kwds={"radius": 6, "fillOpacity": 0.8, "weight": 0.5}
)

In [ ]:
# Split 70/30 into train/test. Splits the classes into train/test in a representative way.
train_gdf, test_gdf = train_test_split(training_data, test_size=0.3, stratify=training_data[class_attr], random_state=42)

print(f"Training set class distribution:\n{train_gdf[class_attr].value_counts()}")
print(f"Test set class distribution:\n{test_gdf[class_attr].value_counts()}")
print(train_gdf)
## Create a classifier and fit a model

# We pass in simple numpy arrays to the classifier, one has the
# observations (the values of the red, green, blue and so on)
# while the other has the classes.
# The classes are the first column
classes = np.array(train_gdf)[:, 0]
print(f"Classes: {classes}")

# The observation data is everything after the first column
# Use train_gdf instead of numpy. A labelled array is safer than a sorted unlabelled one.
observations = train_gdf.drop(columns=class_attr)

# Create a model...
classifier = RandomForestClassifier(class_weight='balanced')

# ...and fit it to the data
model = classifier.fit(observations, classes)
# Define features and target

feature_cols = [c for c in train_gdf.columns if c != class_attr]

X_train = train_gdf[feature_cols].values
y_train = train_gdf[class_attr].values
X_test = test_gdf[feature_cols].values
y_test = test_gdf[class_attr].values

classifier = RandomForestClassifier(n_estimators=500, class_weight="balanced", random_state=42)
model = classifier.fit(X_train, y_train)
y_pred = model.predict(X_test)

# Feature importance — drop noisy features
importances = pd.Series(model.feature_importances_, index=feature_cols).sort_values(ascending=False)
print("Feature importances:")
print(importances)
# Feature importance is probably the most useful next step — it'll tell you which bands are actually helping and which are adding noise.

target_names = [k for k, v in sorted(classes.items(), key=lambda x: x[1]) if v != 0]

print(classification_report(y_test, y_pred, target_names=target_names))
# Actually just don't write the model. Do it all in one notebook.
geomad_dem_indices = get_geomad_dem_indices(country, datetime)

stack = np.stack([geomad_dem_indices[f].values.flatten() for f in feature_cols], axis=1)
stack = np.stack([geomad_dem_indices[f].values.flatten() for f in feature_cols], axis=1)
stack = np.nan_to_num(stack, nan=0.0, posinf=0.0, neginf=0.0).astype(np.float32)

predictions = model.predict(stack)

# Reshape back to raster
prediction_map = predictions.reshape(geomad_dem_indices[feature_cols[0]].shape)

# Wrap in DataArray
predicted_da = xr.DataArray(
    prediction_map,
    coords={"y": geomad_dem_indices.y, "x": geomad_dem_indices.x},
    dims=["y", "x"],
    name="lulc",
)
## Visualise our results

predicted_da = assign_crs(predicted_da, crs="EPSG:6933")

class_indexes = list(colors.keys())
cmap = ListedColormap([colors[c] for c in class_indexes])

predicted_da.odc.explore(categories=class_indexes, cmap=cmap, legend=True, tiles=basemaps.Esri.WorldImagery)

### Aim for >80% accuracy. Don't just look at the confusion matrix, also look at the output map.

# Use a product for validation.
# One validation method for tuning and another for final measure.

target_names = [k for k, v in sorted(classes.items(), key=lambda x: x[1]) if v != 0]

# Classification report
print(classification_report(y_test, y_pred, target_names=target_names))

# Confusion matrix
cm = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=target_names)
disp.plot(xticks_rotation=45, cmap="Blues")